# Single-Cluster 15-Minute Synthetic Stress Test

This notebook imports the reusable generator, builds one planted-feasible 256-node A100 instance from an existing daily energy profile, and solves it with Gurobi. The hidden schedule is a generator certificate and is not passed to Gurobi.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.energy_prices import load_marginalpdbc
from src.data.scenarios import ENERGY_SCENARIOS
from src.data.solar_profiles import build_daily_hourly_solar_profile
from src.instance_generator.single_cluster_stress import (
    SingleClusterStressConfig,
    expand_hourly_energy_to_slots,
    generate_single_cluster_stress_instance,
)

SCENARIO_NAME = 'base'
SEED = 7301
CONFIG = SingleClusterStressConfig(num_jobs=180, node_capacity=256, slot_minutes=15)
INSTANCE_NAME = f'single_cluster_15m_{SCENARIO_NAME}_seed_{SEED}'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'instances' / 'single_cluster_15m' / INSTANCE_NAME
SOLAR_PATH = PROJECT_ROOT / 'data' / 'energy' / 'solar' / 'Timeseries_40.409_-3.698_SA3_1000kWp_crystSi_14_36deg_-5deg_2020_2023.csv'


In [ ]:
scenario = ENERGY_SCENARIOS[SCENARIO_NAME]
price = load_marginalpdbc(PROJECT_ROOT / scenario['price_file'], price_column='last')
solar = build_daily_hourly_solar_profile(SOLAR_PATH, scenario['date'])
hourly = price[['hour', 'grid_price']].merge(solar[['hour', 'renewable_available']], on='hour', validate='one_to_one')
hourly['pue'] = CONFIG.pue
hourly['baseline_load'] = 0.0
hourly['renewable_price'] = 0.8 * float(hourly['grid_price'].min())

jobs_df, clusters_df, energy_df, hidden_schedule, diagnostics_df, metadata = generate_single_cluster_stress_instance(
    expand_hourly_energy_to_slots(hourly, CONFIG.slot_minutes), CONFIG, SEED, INSTANCE_NAME
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
jobs_df.to_csv(OUTPUT_DIR / 'jobs.csv', index=False)
clusters_df.to_csv(OUTPUT_DIR / 'clusters.csv', index=False)
energy_df.to_csv(OUTPUT_DIR / 'energy_inputs_15m.csv', index=False)
hidden_schedule.to_csv(OUTPUT_DIR / 'hidden_feasibility_certificate.csv', index=False)
diagnostics_df.to_csv(OUTPUT_DIR / 'generation_diagnostics.csv', index=False)
(OUTPUT_DIR / 'generation_metadata.json').write_text(json.dumps(metadata, indent=2))

pd.Series(metadata).to_frame('value')


In [ ]:
fig, left = plt.subplots(figsize=(13, 4.5))
left.plot(diagnostics_df['hour'], diagnostics_df['potential_active_nodes'], label='Potential active nodes', color='#d97706')
left.plot(diagnostics_df['hour'], diagnostics_df['hidden_feasible_nodes'], label='Hidden feasible nodes', color='#2563eb')
left.axhline(CONFIG.node_capacity, color='#111827', linestyle='--', label='Capacity')
left.set(xlabel='15-minute slot', ylabel='Nodes')
left.grid(alpha=0.25)
right = left.twinx()
right.plot(diagnostics_df['hour'], diagnostics_df['effective_price'], label='Effective price', color='#15803d')
right.set_ylabel('Effective price')
handles, labels = left.get_legend_handles_labels()
right_handles, right_labels = right.get_legend_handles_labels()
left.legend(handles + right_handles, labels + right_labels, loc='upper left')
plt.show()


## Gurobi validation

Gurobi receives only the generated jobs, the 256-node capacity, and 15-minute energy inputs. CPU and memory are disabled because this benchmark isolates node capacity and energy timing.


In [ ]:
import time
from src.config import ModelConfig
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model

model_config = ModelConfig(
    contracted_power=float(clusters_df['capacity'].sum()) * CONFIG.pue,
    renewable_price=float(energy_df['renewable_price'].iloc[0]),
    peak_price=1000.0,
    delta_t=CONFIG.delta_t,
    pue=CONFIG.pue,
)
milp_energy = energy_df[['hour', 'renewable_available', 'grid_price', 'pue', 'baseline_load']]
started = time.perf_counter()
model, variables = build_milp_model(
    jobs_df, milp_energy, clusters_df, model_config,
    model_name=INSTANCE_NAME,
    enforce_gpu_constraints=True,
    enforce_cpu_constraints=False,
    enforce_memory_constraints=False,
)
model.Params.OutputFlag = 0
solve_model(model, time_limit=180, mip_gap=0.001)
summary = pd.DataFrame([{
    'status': model.Status,
    'has_solution': bool(model.SolCount),
    'runtime_s': time.perf_counter() - started,
    'mip_gap': model.MIPGap if model.SolCount else None,
    'objective': model.ObjVal if model.SolCount else None,
    'best_bound': model.ObjBound if model.SolCount else None,
    'assignment_variables': len(variables['x']),
    'constraints': model.NumConstrs,
}])
summary.to_csv(OUTPUT_DIR / 'gurobi_summary.csv', index=False)
summary
